# Stage 2 — CREsted enhancer code analysis, full 100 topics

This notebook follows the CREsted **Enhancer code analysis** tutorial structure as closely as possible, adapted to your project paths and your `topic × region` AnnData object.

Main idea:

1. Load genome, AnnData, and trained topic classification model  
2. Predict all regions  
3. Store model predictions in `adata.layers["model_prediction"]`  
4. Build `adata.layers["combined"] = (adata.X + model_prediction) / 2`  
5. Use `crested.pp.sort_and_filter_regions_on_specificity()` to keep the top specific regions per topic  
6. Run `crested.tl.contribution_scores_specific()` for all 100 topics  
7. Run `crested.tl.modisco.tfmodisco()` on contribution scores  
8. Export topic → celltype annotation bridge for later interpretation

This is the **full 100-topic version**. It may be heavy.

## 0. Notes before running

This notebook intentionally does **not** filter topics first. It runs all 100 topics.

Recommended first run:

- `SPECIES = "human"`
- `BATCH_SIZE_PREDICT = 4`
- `TOP_K = 2000`
- `CONTRIB_METHOD = "integrated_grad"`

If it is too slow or memory-heavy, reduce:

- `TOP_K = 500`
- `CONTRIB_BATCH_SIZE = 16`
- `MAX_SEQLETS = 10000`

The CREsted tutorial computes model predictions, creates a combined layer from ground truth and predictions, filters class-specific regions with `sort_and_filter_regions_on_specificity`, computes class-specific contribution scores, and then runs tfmodisco-lite.

In [1]:
import os
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
import anndata as ad

import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
import matplotlib.pyplot as plt

os.environ["KERAS_BACKEND"] = "torch"

import keras
import crested

## 1. Configuration

In [2]:
SPECIES = "human"  # "human" or "macaque"

BASE = Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt")

CONFIG = {
    "human": {
        "adata_path": BASE / "runs" / "out" / "human_topics.h5ad",
        "model_path": BASE / "runs" / "out" / "deeptopic_human" / "final_model.keras",
        "topic_annotation_path": BASE / "data" / "stage0_annotation" / "topic_annotation_human.tsv",
        "genome_fa": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/genomes/hg38/hg38.fa"),
        # chrom sizes optional; set to None if unavailable
        "chrom_sizes": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/genomes/hg38/hg38.chrom.sizes"),
    },
    "macaque": {
        "adata_path": BASE / "runs" / "out" / "macaque_topics.h5ad",
        "model_path": BASE / "runs" / "out" / "deeptopic_macaque" / "final_model.keras",
        "topic_annotation_path": BASE / "data" / "stage0_annotation" / "topic_annotation_macaque.tsv",
        "genome_fa": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/yiquan/macaque/rheMac10.fa"),
        "chrom_sizes": None,
    },
}

cfg = CONFIG[SPECIES]

OUTDIR = BASE / "runs" / "out" / "stage2_enhancer_code_full100" / SPECIES
OUTDIR.mkdir(parents=True, exist_ok=True)

PRED_LAYER = "model_prediction"
COMBINED_LAYER = "combined"

# Directly following the tutorial idea: top class-specific regions per topic
TOP_K = 2000
SPECIFICITY_METHOD = "gini"

# prediction / contribution settings
BATCH_SIZE_PREDICT = 4
CONTRIB_BATCH_SIZE = 32
CONTRIB_METHOD = "integrated_grad"  # "expected_integrated_grad" is heavier; tutorial says IG is faster and often similar

# tfmodisco settings
RUN_TFMODISCO = True
MAX_SEQLETS = 20000
MODISCO_WINDOW = 1000

# Motif DB:
# If you already have a MEME motif database, put its path here.
# If None, the tfmodisco step will try crested.get_motif_db("jaspar").
MEME_DB = None

print("SPECIES:", SPECIES)
print("OUTDIR:", OUTDIR)
print("adata:", cfg["adata_path"])
print("model:", cfg["model_path"])
print("genome:", cfg["genome_fa"])
print("topic annotation:", cfg["topic_annotation_path"])

SPECIES: human
OUTDIR: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage2_enhancer_code_full100/human
adata: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/human_topics.h5ad
model: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/deeptopic_human/final_model.keras
genome: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/genomes/hg38/hg38.fa
topic annotation: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/stage0_annotation/topic_annotation_human.tsv


## 2. Load genome, AnnData, model, and topic annotation

In [3]:
adata = ad.read_h5ad(cfg["adata_path"])
topic_anno = pd.read_csv(cfg["topic_annotation_path"], sep="\t")

# Load model
try:
    model = crested.utils.load_model(str(cfg["model_path"]))
except Exception:
    model = keras.models.load_model(str(cfg["model_path"]), compile=False)

# Register genome, following CREsted tutorial style
chrom_sizes = cfg.get("chrom_sizes", None)
if chrom_sizes is not None and Path(chrom_sizes).exists():
    genome = crested.Genome(str(cfg["genome_fa"]), str(chrom_sizes))
else:
    genome = crested.Genome(str(cfg["genome_fa"]))

crested.register_genome(genome)

print(adata)
print("obs/topics:", adata.obs.index[:5].tolist())
print("var/regions:", adata.var.index[:5].tolist())
print("var columns:", list(adata.var.columns))
display(topic_anno.head())

2026-04-27T15:03:07.398521+0200 INFO Genome hg38 registered.
AnnData object with n_obs × n_vars = 100 × 415405
    obs: 'file_path', 'n_open_regions'
    var: 'n_classes', 'chr', 'start', 'end', 'split'
obs/topics: ['Topic1', 'Topic10', 'Topic100', 'Topic11', 'Topic12']
var/regions: ['chr1:9973-10473', 'chr1:180766-181266', 'chr1:191161-191661', 'chr1:628999-629499', 'chr1:629674-630174']
var columns: ['n_classes', 'chr', 'start', 'end', 'split']


,topic,topic_num,species,top1_celltype,top1_score,top2_celltype,top2_score,specificity_ratio_top1_over_top2,score_delta_top1_minus_top2,annotation_confidence,annotation_class,lineage_group,lineage_group_top2,bed_path,bed_exists,n_peaks,usable_for_motif
0,Topic1,1,human,OPC,0.018942,Vascular,0.016960,1.116855,0.001982,low,ambiguous,glia,vascular,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic1.bed,True,8150,False
1,Topic2,2,human,CGE_interneuron,0.011338,LGE_FOXP2_TSHZ1_MSN,0.010950,1.035347,0.000387,low,ambiguous,interneuron,subpallial_neuron,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic2.bed,True,6984,False
2,Topic3,3,human,OPC,0.015154,ExNeu_IT,0.012243,1.237767,0.002911,medium,ambiguous,glia,excitatory_neuron,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic3.bed,True,15275,False
3,Topic4,4,human,ExIPC,0.017248,MGE_progenitors,0.013981,1.233697,0.003267,medium,ambiguous,progenitor,progenitor,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic4.bed,True,4338,False
4,Topic5,5,human,ExNeu_IT,0.027088,ExNeu_Non_IT,0.015738,1.721155,0.011350,high,clean_top1,excitatory_neuron,excitatory_neuron,/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/topic_beds_human_otsu/Topic5.bed,True,10672,True


## 3. Sanity checks

Your object should be:

- `adata.obs` = 100 topics
- `adata.var` = genomic regions
- `adata.X` = topic-region matrix

In [4]:
assert adata.n_obs == 100, f"Expected 100 topics/classes in obs, got {adata.n_obs}"
assert "chr" in adata.var.columns and "start" in adata.var.columns and "end" in adata.var.columns, "adata.var must contain chr/start/end"
print("adata shape:", adata.shape)
print("n topics:", adata.n_obs)
print("n regions:", adata.n_vars)
print("X shape:", adata.X.shape)

adata shape: (100, 415405)
n topics: 100
n regions: 415405
X shape: (100, 415405)


## 4. Predict all regions

This follows the tutorial logic: calculate predictions and store them as an AnnData layer.

Important shape note:

- CREsted prediction output is `n_regions × n_topics`
- AnnData layers require `n_topics × n_regions`
- Therefore we transpose before storing:
  `adata.layers["model_prediction"] = predictions.T`

In [5]:
pred_cache = OUTDIR / "all_region_predictions.npy"

if pred_cache.exists():
    print("Loading cached predictions:", pred_cache)
    predictions = np.load(pred_cache)
else:
    predictions = crested.tl.predict(
        adata,
        model,
        genome=genome,
        batch_size=BATCH_SIZE_PREDICT,
    )
    if not isinstance(predictions, np.ndarray):
        predictions = np.asarray(predictions)

    np.save(pred_cache, predictions)
    print("Saved prediction cache:", pred_cache)

print("predictions shape:", predictions.shape)

expected_shape = (adata.n_vars, adata.n_obs)
if predictions.shape != expected_shape:
    raise ValueError(f"Expected prediction shape {expected_shape}, got {predictions.shape}")

adata.layers[PRED_LAYER] = predictions.T
print("Stored layer:", PRED_LAYER, adata.layers[PRED_LAYER].shape)

103852/103852 ━━━━━━━━━━━━━━━━━━━━ 1495s 14ms/step
Saved prediction cache: /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage2_enhancer_code_full100/human/all_region_predictions.npy
predictions shape: (415405, 100)
Stored layer: model_prediction (100, 415405)


## 5. Create combined layer

Following the CREsted tutorial, combine ground truth topic-region values and model predictions:

`combined = (adata.X + adata.layers["model_prediction"]) / 2`

This layer is then used to choose regions that are both topic-specific in the original data and supported by the sequence model.

In [ ]:
X = adata.X
if hasattr(X, "toarray"):
    X_dense = X.toarray()
else:
    X_dense = np.asarray(X)

adata.layers[COMBINED_LAYER] = (X_dense + adata.layers[PRED_LAYER]) / 2

print("combined layer:", adata.layers[COMBINED_LAYER].shape)
print("combined min/max:", np.nanmin(adata.layers[COMBINED_LAYER]), np.nanmax(adata.layers[COMBINED_LAYER]))

## 6. QC plot: specificity cutoff

The tutorial uses CREsted QC plots to inspect specificity / Gini score distribution before selecting top regions.
If this plotting function is unavailable in your CREsted version, this cell will skip gracefully.

In [ ]:
try:
    crested.pl.qc.sort_and_filter_cutoff(
        adata,
        model_name=COMBINED_LAYER,
        method=SPECIFICITY_METHOD,
    )
    plt.savefig(OUTDIR / "qc_sort_and_filter_cutoff.png", dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved QC plot")
except Exception as e:
    print("QC plot skipped:", repr(e))

## 7. Sort and filter top specific regions per topic

This is the tutorial's key preprocessing step:

`crested.pp.sort_and_filter_regions_on_specificity(...)`

Here we keep the top `TOP_K` regions per topic, using the `combined` layer.

In [ ]:
adata_filtered_path = OUTDIR / f"adata_filtered_top{TOP_K}_{SPECIFICITY_METHOD}.h5ad"

if adata_filtered_path.exists():
    print("Loading cached filtered AnnData:", adata_filtered_path)
    adata_filtered = ad.read_h5ad(adata_filtered_path)
else:
    adata_filtered = crested.pp.sort_and_filter_regions_on_specificity(
        adata,
        model_name=COMBINED_LAYER,
        top_k=TOP_K,
        method=SPECIFICITY_METHOD,
        inplace=False,
    )
    adata_filtered.write_h5ad(adata_filtered_path)
    print("Saved filtered AnnData:", adata_filtered_path)

print(adata_filtered)
print("filtered shape:", adata_filtered.shape)
print("var columns:", list(adata_filtered.var.columns))
display(adata_filtered.var.head())

## 8. Save selected regions per topic

This makes it easier to inspect which regions are going into contribution scoring for each topic.

In [ ]:
selected_dir = OUTDIR / "selected_regions_per_topic"
selected_dir.mkdir(exist_ok=True)

# CREsted filtered object usually has a Class name column in var for contribution_scores_specific.
# But we save a general region table anyway.
region_table = adata_filtered.var.copy()
region_table["region"] = region_table.index.astype(str)
region_table.to_csv(OUTDIR / f"selected_regions_top{TOP_K}_all_topics.tsv", sep="\t", index=False)

print("Saved selected region table:", OUTDIR / f"selected_regions_top{TOP_K}_all_topics.tsv")

# Also export topic annotation bridge
topic_anno.to_csv(OUTDIR / "topic_annotation_bridge.tsv", sep="\t", index=False)
print("Saved topic annotation bridge")

## 9. Contribution scores for all 100 topics

This directly follows the tutorial step with `crested.tl.contribution_scores_specific`.

`target_idx=None` means all classes/topics.

Output directory will contain one `.npz` file per topic/class.

In [ ]:
contrib_dir = OUTDIR / f"contribution_scores_{CONTRIB_METHOD}_top{TOP_K}"
contrib_dir.mkdir(exist_ok=True)

print("Contribution output:", contrib_dir)

contrib_scores, one_hot_seqs = crested.tl.contribution_scores_specific(
    input=adata_filtered,
    target_idx=None,
    model=model,
    genome=genome,
    method=CONTRIB_METHOD,
    batch_size=CONTRIB_BATCH_SIZE,
    output_dir=str(contrib_dir),
    verbose=True,
)

print("Contribution scores returned.")
try:
    print("contrib_scores shape:", contrib_scores.shape)
except Exception as e:
    print("contrib_scores shape unavailable:", e)
try:
    print("one_hot_seqs shape:", one_hot_seqs.shape)
except Exception as e:
    print("one_hot_seqs shape unavailable:", e)

## 10. Run tfmodisco-lite

This follows the tutorial's `crested.tl.modisco.tfmodisco(...)` step.

If you do not have a motif DB path, this notebook tries to fetch a JASPAR motif DB using CREsted.
If fetching fails on HPC, set `MEME_DB` manually in the config cell.

In [ ]:
modisco_dir = OUTDIR / f"tfmodisco_top{TOP_K}_{CONTRIB_METHOD}"
modisco_dir.mkdir(exist_ok=True)

if RUN_TFMODISCO:
    meme_db = MEME_DB

    if meme_db is None:
        try:
            meme_db = crested.get_motif_db("jaspar")
            print("Using motif DB from crested.get_motif_db('jaspar'):", meme_db)
        except Exception as e:
            print("Could not fetch motif DB automatically:", repr(e))
            print("Set MEME_DB manually in the config cell and rerun this cell.")
            meme_db = None

    if meme_db is not None:
        crested.tl.modisco.tfmodisco(
            window=MODISCO_WINDOW,
            output_dir=str(modisco_dir),
            contrib_dir=str(contrib_dir),
            report=False,
            meme_db=meme_db,
            max_seqlets=MAX_SEQLETS,
        )
        print("tfmodisco finished:", modisco_dir)
    else:
        print("tfmodisco skipped because meme_db is None.")
else:
    print("RUN_TFMODISCO=False, skipped tfmodisco.")

## 11. Save run metadata

In [ ]:
metadata = {
    "species": SPECIES,
    "adata_path": str(cfg["adata_path"]),
    "model_path": str(cfg["model_path"]),
    "genome_fa": str(cfg["genome_fa"]),
    "topic_annotation_path": str(cfg["topic_annotation_path"]),
    "outdir": str(OUTDIR),
    "prediction_layer": PRED_LAYER,
    "combined_layer": COMBINED_LAYER,
    "top_k": TOP_K,
    "specificity_method": SPECIFICITY_METHOD,
    "batch_size_predict": BATCH_SIZE_PREDICT,
    "contrib_batch_size": CONTRIB_BATCH_SIZE,
    "contrib_method": CONTRIB_METHOD,
    "run_tfmodisco": RUN_TFMODISCO,
    "max_seqlets": MAX_SEQLETS,
    "modisco_window": MODISCO_WINDOW,
}

with open(OUTDIR / "stage2_full100_run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

## 12. What to inspect after running

Important outputs:

- `all_region_predictions.npy`
- `adata_filtered_top{TOP_K}_{SPECIFICITY_METHOD}.h5ad`
- `selected_regions_top{TOP_K}_all_topics.tsv`
- `topic_annotation_bridge.tsv`
- `contribution_scores_{CONTRIB_METHOD}_top{TOP_K}/`
- `tfmodisco_top{TOP_K}_{CONTRIB_METHOD}/`

Next step after this notebook:

Use `topic_annotation_bridge.tsv` to translate:

`Topic-level motif patterns → celltype-associated motif sets`